# Direct Data Access Guide for PBI

This notebook demonstrates efficient bulk data retrieval using direct database access, avoiding the REST API overhead.

## Key Benefits

- **5-50x faster** than API for bulk operations
- **Direct access** to DuckDB database and FASTA files
- **Batch processing** support for large datasets
- **Read-only access** prevents accidental data corruption
- **No network overhead** - file system access only

## Architecture

```
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│   Pipeline   │────▶│  pbi-data    │◀────│   Analysis   │
│  (writes)    │     │   volume     │     │  (read-only) │
└──────────────┘     └──────────────┘     └──────────────┘
                           │
                           │
                     ┌─────┴─────┐
                     │    API    │
                     │(read-only)│
                     └───────────┘
```

## Table of Contents

1. [Setup and Connection](#setup)
2. [Basic Metadata Queries](#basic-queries)
3. [Sequence Retrieval](#sequence-retrieval)
4. [Batch Processing](#batch-processing)
5. [Data Export](#data-export)
6. [Real-World Example: E. coli Phages Analysis](#ecoli-example)
7. [Performance Comparison](#performance)
8. [Best Practices](#best-practices)

## 1. Setup and Connection <a id="setup"></a>

First, let's verify our environment and establish connections.

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from pbi import SequenceRetriever
import time
from datetime import datetime

# Set display options for better output
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Imports successful")
print(f"Timestamp: {datetime.now()}")

In [ ]:
# Define data paths (Docker environment)
DATA_PATH = Path("/data/processed")
DB_PATH = DATA_PATH / "databases" / "phage_database_optimized.duckdb"
SEQUENCES_PATH = DATA_PATH / "sequences"

PHAGE_FASTA = SEQUENCES_PATH / "all_phages.fasta"
PROTEIN_FASTA = SEQUENCES_PATH / "all_proteins.fasta"
HOST_MAPPING = SEQUENCES_PATH / "host_fasta_mapping.json"

# Verify paths exist
print("📂 Verifying data paths:")
print(f"  Database: {DB_PATH.exists()} ({DB_PATH})")
print(f"  Phage FASTA: {PHAGE_FASTA.exists()} ({PHAGE_FASTA})")
print(f"  Protein FASTA: {PROTEIN_FASTA.exists()} ({PROTEIN_FASTA})")
print(f"  Host Mapping: {HOST_MAPPING.exists()} ({HOST_MAPPING})")

if DB_PATH.exists():
    print(f"\n📊 Database size: {DB_PATH.stat().st_size / 1e9:.2f} GB")

### Important: Always Use Read-Only Connections

⚠️ **Critical Best Practice**: Always connect to DuckDB with `read_only=True` to prevent:
- Accidental data modification
- Database lock conflicts
- Data corruption

The data volume is mounted read-only (`:ro`), but DuckDB needs explicit read-only flag.

In [ ]:
# Connect to DuckDB (READ-ONLY)
# Using context manager ensures proper cleanup
conn = duckdb.connect(str(DB_PATH), read_only=True)

print("✅ Connected to DuckDB (read-only mode)")

# Quick database info
tables = conn.execute("SHOW TABLES").fetchdf()
print(f"\n📊 Available tables: {len(tables)}")
print(tables)

In [ ]:
# Initialize SequenceRetriever for FASTA access
retriever = SequenceRetriever(
    db_path=str(DB_PATH),
    phage_fasta_path=str(PHAGE_FASTA),
    protein_fasta_path=str(PROTEIN_FASTA),
    host_mapping_path=str(HOST_MAPPING) if HOST_MAPPING.exists() else None,
    preload=True  # Load FASTA indexes in background
)

print("✅ SequenceRetriever initialized")
print(f"   Host data available: {retriever.has_host_data()}")

## 2. Basic Metadata Queries <a id="basic-queries"></a>

Direct SQL queries are extremely fast for metadata retrieval.

In [ ]:
# Get database statistics
stats_query = """
SELECT 
    'Phages' as Entity,
    COUNT(*) as Count
FROM fact_phages
UNION ALL
SELECT 
    'Proteins',
    COUNT(*)
FROM fact_proteins
UNION ALL
SELECT 
    'Host Associations',
    COUNT(*)
FROM fact_phage_host
"""

stats_df = conn.execute(stats_query).fetchdf()
print("📊 Database Statistics:")
print(stats_df.to_string(index=False))

In [ ]:
# Example: Query phages by criteria
query = """
SELECT 
    Phage_ID,
    Accession,
    Length,
    GC_Content,
    Completeness,
    Database
FROM fact_phages
WHERE Length > 100000  -- Large phages
  AND Completeness = 'complete'
  AND GC_Content IS NOT NULL
ORDER BY Length DESC
LIMIT 10
"""

start = time.time()
large_phages = conn.execute(query).fetchdf()
query_time = time.time() - start

print(f"⚡ Query completed in {query_time*1000:.2f} ms")
print("\n🔬 Top 10 largest complete phages:")
print(large_phages)

## 3. Sequence Retrieval <a id="sequence-retrieval"></a>

Combine metadata queries with sequence retrieval using the SequenceRetriever.

In [ ]:
# Get sequences for the largest phages
phage_ids = large_phages['Phage_ID'].tolist()[:5]  # First 5 for demo

print("🧬 Retrieving sequences...")
sequences = retriever.get_sequences_by_ids(phage_ids, sequence_type='phage')

print(f"\n✅ Retrieved {len(sequences)} sequences")
for phage_id, seq in list(sequences.items())[:3]:
    print(f"  {phage_id}: {len(seq)} bp (first 60 bp: {seq[:60]}...)")

## 4. Batch Processing <a id="batch-processing"></a>

Process large datasets in chunks to avoid memory issues.

### Memory-Efficient Pattern: Process in Batches

In [ ]:
def process_phages_in_batches(conn, batch_size=1000, max_batches=None):
    """
    Process phages in batches to avoid out-of-memory errors.
    
    This pattern is essential for working with millions of records.
    """
    # Get total count
    total = conn.execute("SELECT COUNT(*) FROM fact_phages").fetchone()[0]
    
    print(f"📊 Processing {total:,} phages in batches of {batch_size:,}")
    
    results = []
    offset = 0
    batch_num = 0
    
    while True:
        # Fetch batch
        query = f"""
        SELECT 
            Phage_ID,
            Length,
            GC_Content
        FROM fact_phages
        WHERE GC_Content IS NOT NULL
        LIMIT {batch_size} OFFSET {offset}
        """
        
        batch = conn.execute(query).fetchdf()
        
        if len(batch) == 0:
            break
        
        # Process batch (example: calculate statistics)
        batch_stats = {
            'batch': batch_num,
            'count': len(batch),
            'avg_length': batch['Length'].mean(),
            'avg_gc': batch['GC_Content'].mean(),
        }
        results.append(batch_stats)
        
        offset += batch_size
        batch_num += 1
        
        if max_batches and batch_num >= max_batches:
            break
        
        if batch_num % 10 == 0:
            print(f"  Processed {offset:,} records...")
    
    return pd.DataFrame(results)

# Demo: Process first 5 batches
batch_results = process_phages_in_batches(conn, batch_size=1000, max_batches=5)
print("\n📈 Batch processing results:")
print(batch_results)

### Streaming Results from DuckDB

For very large result sets, use DuckDB's fetch_arrow_table for memory efficiency.

In [ ]:
# Example: Aggregate statistics without loading all data into memory
streaming_query = """
SELECT 
    Database,
    COUNT(*) as Phage_Count,
    AVG(Length) as Avg_Length,
    AVG(GC_Content) as Avg_GC,
    MIN(Length) as Min_Length,
    MAX(Length) as Max_Length
FROM fact_phages
WHERE GC_Content IS NOT NULL
GROUP BY Database
ORDER BY Phage_Count DESC
"""

start = time.time()
db_stats = conn.execute(streaming_query).fetchdf()
query_time = time.time() - start

print(f"⚡ Aggregated millions of records in {query_time:.3f} seconds")
print("\n📊 Statistics by database:")
print(db_stats)

## 5. Data Export <a id="data-export"></a>

Export large datasets efficiently using DuckDB's native capabilities.

In [ ]:
# Create output directory
output_dir = Path("/workspace/exports")
output_dir.mkdir(exist_ok=True)

print(f"📁 Export directory: {output_dir}")

In [ ]:
# Export to Parquet (most efficient for large datasets)
export_query = """
COPY (
    SELECT 
        p.Phage_ID,
        p.Accession,
        p.Length,
        p.GC_Content,
        p.Database,
        ph.Host_ID,
        ph.Host_Name
    FROM fact_phages p
    LEFT JOIN fact_phage_host ph ON p.Phage_ID = ph.Phage_ID
    WHERE p.Length > 50000
) TO '/workspace/exports/large_phages.parquet' (FORMAT PARQUET)
"""

start = time.time()
conn.execute(export_query)
export_time = time.time() - start

parquet_file = output_dir / "large_phages.parquet"
file_size = parquet_file.stat().st_size / 1e6 if parquet_file.exists() else 0

print(f"✅ Exported to Parquet in {export_time:.2f} seconds")
print(f"   File size: {file_size:.2f} MB")

In [ ]:
# Export to CSV (for compatibility)
csv_query = """
COPY (
    SELECT * FROM fact_phages
    WHERE Completeness = 'complete'
    LIMIT 10000
) TO '/workspace/exports/complete_phages_sample.csv' 
(HEADER, DELIMITER ',')
"""

start = time.time()
conn.execute(csv_query)
export_time = time.time() - start

print(f"✅ Exported 10,000 records to CSV in {export_time:.2f} seconds")

## 6. Real-World Example: E. coli Phages Analysis <a id="ecoli-example"></a>

A complete workflow demonstrating:
1. Query phages targeting E. coli
2. Retrieve their sequences
3. Analyze genomic features
4. Export results

In [ ]:
# Step 1: Find E. coli phages
ecoli_query = """
SELECT DISTINCT
    p.Phage_ID,
    p.Accession,
    p.Length,
    p.GC_Content,
    p.Completeness,
    ph.Host_Name,
    ph.Host_Genus
FROM fact_phages p
INNER JOIN fact_phage_host ph ON p.Phage_ID = ph.Phage_ID
WHERE ph.Host_Genus LIKE '%Escherichia%'
  OR ph.Host_Name LIKE '%coli%'
ORDER BY p.Length DESC
"""

print("🔍 Querying E. coli phages...")
start = time.time()
ecoli_phages = conn.execute(ecoli_query).fetchdf()
query_time = time.time() - start

print(f"✅ Found {len(ecoli_phages):,} E. coli phages in {query_time*1000:.2f} ms")
print("\n📊 Summary:")
print(ecoli_phages.head(10))

In [ ]:
# Step 2: Analyze genome length distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Length distribution
axes[0].hist(ecoli_phages['Length'] / 1000, bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Genome Length (kb)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('E. coli Phage Genome Length Distribution')
axes[0].axvline(ecoli_phages['Length'].median() / 1000, color='red', 
                linestyle='--', label=f'Median: {ecoli_phages["Length"].median()/1000:.1f} kb')
axes[0].legend()

# GC content distribution
ecoli_phages_gc = ecoli_phages.dropna(subset=['GC_Content'])
axes[1].hist(ecoli_phages_gc['GC_Content'] * 100, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('GC Content (%)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('E. coli Phage GC Content Distribution')
axes[1].axvline(ecoli_phages_gc['GC_Content'].median() * 100, color='red',
                linestyle='--', label=f'Median: {ecoli_phages_gc["GC_Content"].median()*100:.1f}%')
axes[1].legend()

plt.tight_layout()
plt.savefig(output_dir / 'ecoli_phage_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("📊 Plot saved to exports/ecoli_phage_analysis.png")

In [ ]:
# Step 3: Retrieve sequences for a subset (memory-efficient)
sample_size = min(100, len(ecoli_phages))
sample_phages = ecoli_phages.sample(n=sample_size, random_state=42)

print(f"🧬 Retrieving sequences for {sample_size} E. coli phages...")
start = time.time()

phage_ids = sample_phages['Phage_ID'].tolist()
sequences = retriever.get_sequences_by_ids(phage_ids, sequence_type='phage')

retrieval_time = time.time() - start

print(f"✅ Retrieved {len(sequences)} sequences in {retrieval_time:.2f} seconds")
print(f"   Average time per sequence: {retrieval_time/len(sequences)*1000:.2f} ms")

In [ ]:
# Step 4: Calculate sequence statistics
from Bio.SeqUtils import gc_fraction
from Bio.Seq import Seq

seq_stats = []
for phage_id, seq_str in sequences.items():
    seq = Seq(seq_str)
    seq_stats.append({
        'Phage_ID': phage_id,
        'Sequence_Length': len(seq),
        'Calculated_GC': gc_fraction(seq),
        'A_count': seq_str.count('A'),
        'T_count': seq_str.count('T'),
        'G_count': seq_str.count('G'),
        'C_count': seq_str.count('C'),
    })

seq_stats_df = pd.DataFrame(seq_stats)

# Merge with metadata
ecoli_results = sample_phages.merge(seq_stats_df, on='Phage_ID')

print("📊 Sequence statistics calculated")
print(ecoli_results[['Phage_ID', 'Length', 'Sequence_Length', 'GC_Content', 'Calculated_GC']].head())

In [ ]:
# Step 5: Export results
output_file = output_dir / 'ecoli_phage_analysis.csv'
ecoli_results.to_csv(output_file, index=False)

print(f"✅ Results exported to {output_file}")
print(f"   Records: {len(ecoli_results)}")
print(f"   Columns: {len(ecoli_results.columns)}")
print(f"   File size: {output_file.stat().st_size / 1024:.2f} KB")

## 7. Performance Comparison <a id="performance"></a>

Compare direct database access vs API approach.

In [ ]:
import requests

def benchmark_direct_vs_api(num_records=100):
    """
    Compare performance of direct access vs API.
    
    Note: API must be running for this comparison.
    """
    results = {}
    
    # Benchmark 1: Direct SQL query
    query = f"SELECT * FROM fact_phages LIMIT {num_records}"
    
    start = time.time()
    direct_df = conn.execute(query).fetchdf()
    results['direct_query_time'] = time.time() - start
    results['direct_records'] = len(direct_df)
    
    # Benchmark 2: API query (if available)
    api_url = "http://api:8000"  # Docker network
    
    try:
        start = time.time()
        response = requests.get(f"{api_url}/phages", params={"limit": num_records}, timeout=30)
        results['api_query_time'] = time.time() - start
        
        if response.status_code == 200:
            api_data = response.json()
            results['api_records'] = len(api_data.get('phages', []))
            results['speedup'] = results['api_query_time'] / results['direct_query_time']
        else:
            results['api_error'] = f"Status {response.status_code}"
    except Exception as e:
        results['api_error'] = str(e)
        print(f"⚠️  API not available: {e}")
    
    return results

print("⚡ Running performance benchmark...")
benchmark_results = benchmark_direct_vs_api(num_records=1000)

print("\n📊 Performance Comparison:")
print(f"  Direct query: {benchmark_results['direct_query_time']*1000:.2f} ms ({benchmark_results['direct_records']} records)")

if 'api_query_time' in benchmark_results:
    print(f"  API query: {benchmark_results['api_query_time']*1000:.2f} ms ({benchmark_results['api_records']} records)")
    print(f"  ⚡ Speedup: {benchmark_results['speedup']:.1f}x faster with direct access")
else:
    print(f"  API: {benchmark_results.get('api_error', 'Not available')}")
    print("  Note: Start the API service to compare performance")

## 8. Best Practices <a id="best-practices"></a>

### ✅ DO:

1. **Always use read-only connections**
   ```python
   conn = duckdb.connect(db_path, read_only=True)
   ```

2. **Process data in batches**
   ```python
   # Good: Process 1000 at a time
   LIMIT 1000 OFFSET {offset}
   
   # Bad: Load millions of records at once
   SELECT * FROM fact_phages  # 800K+ records!
   ```

3. **Use DuckDB's export functions**
   ```python
   COPY (...) TO 'output.parquet' (FORMAT PARQUET)
   ```

4. **Close connections properly**
   ```python
   try:
       conn = duckdb.connect(db_path, read_only=True)
       # ... work ...
   finally:
       conn.close()
   ```

5. **Use aggregations instead of loading all data**
   ```python
   # Good: Calculate statistics in database
   SELECT AVG(Length), STD(Length) FROM fact_phages
   
   # Bad: Load all data to calculate in Python
   df = conn.execute("SELECT Length FROM fact_phages").fetchdf()
   mean = df['Length'].mean()  # Unnecessary memory usage
   ```

### ❌ DON'T:

1. **Don't open writable connections**
   ```python
   # Bad: Could corrupt data
   conn = duckdb.connect(db_path)  # Missing read_only=True
   ```

2. **Don't load entire tables into memory**
   ```python
   # Bad: 800K+ rows could crash kernel
   all_phages = conn.execute("SELECT * FROM fact_phages").fetchdf()
   ```

3. **Don't retrieve sequences for millions of phages at once**
   ```python
   # Bad: Will run out of memory
   all_ids = conn.execute("SELECT Phage_ID FROM fact_phages").fetchdf()
   sequences = retriever.get_sequences_by_ids(all_ids['Phage_ID'].tolist())
   
   # Good: Process in batches
   batch_size = 1000
   for i in range(0, len(all_ids), batch_size):
       batch_ids = all_ids.iloc[i:i+batch_size]['Phage_ID'].tolist()
       sequences = retriever.get_sequences_by_ids(batch_ids)
       # Process batch...
   ```

4. **Don't forget to filter data**
   ```python
   # Good: Filter in SQL
   WHERE Length > 50000 AND Completeness = 'complete'
   
   # Bad: Load everything then filter in pandas
   df = df[(df['Length'] > 50000) & (df['Completeness'] == 'complete')]
   ```

## Common Issues and Solutions

### Issue 1: "Database is locked"
**Solution**: Always use `read_only=True` when connecting to DuckDB.

### Issue 2: Out of Memory Error
**Solution**: Process data in smaller batches (e.g., 1000-10000 records at a time).

### Issue 3: Slow Sequence Retrieval
**Solution**: 
- FASTA files must be indexed (`.fai` files)
- Retrieve sequences in batches, not one at a time
- Use `pyfaidx` indexed access (already done by SequenceRetriever)

### Issue 4: Path Not Found Errors
**Solution**: 
- In Docker: Use `/data/processed/*` paths
- Locally: Set `DATA_PATH` environment variable or use relative paths

### Issue 5: Kernel Crashes
**Solution**: 
- Reduce batch size
- Use SQL aggregations instead of loading raw data
- Monitor memory usage: `!free -h` or `!htop`

## Cleanup

Always close connections when done.

In [ ]:
# Close DuckDB connection
if 'conn' in locals():
    conn.close()
    print("✅ Database connection closed")

# SequenceRetriever cleanup is automatic
print("✅ All resources released")

## Summary

This notebook demonstrated:

✅ **Direct database access** - 5-50x faster than API  
✅ **Read-only safety** - Prevents data corruption  
✅ **Batch processing** - Handle millions of records efficiently  
✅ **Efficient exports** - Use native DuckDB export functions  
✅ **Real-world workflow** - Complete E. coli phage analysis  
✅ **Best practices** - Memory management and performance optimization  

## Next Steps

- Explore other tables: `fact_proteins`, `fact_phage_host`
- Join multiple tables for complex analyses
- Build machine learning datasets
- Export data for external tools (R, BLAST, etc.)
- Create custom visualizations

## Resources

- [PBI Documentation](https://thibaultschowing.github.io/PBI/)
- [DuckDB Documentation](https://duckdb.org/docs/)
- [pyfaidx Documentation](https://github.com/mdshw5/pyfaidx)
- [Docker Guide](https://thibaultschowing.github.io/PBI/guides/docker-guide/)